# Modeling data from Liu et al.'s paper

This notebook models the data from this [paper](https://jitc.bmj.com/content/10/12/e005360).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Defining and simulating the model

Here we will be setting up and simulating the differential equations using ```scipy.integrate.odeint```. From the paper there are 4 models to construct:
- CD19+ B-ALL Cell
- CAR T-cell activation
- Non-Activated CAR T-cell
- CD19+ and CD19- relapse

In [2]:
def cd19_b_all_cell(np_cells, nTA, params):
    """
    Calculates the derivative of CD19⁺ B-ALL cells.
    """
    r_p, n_c, e, K_p = params
    dnp_dt = r_p * (1 - np_cells / n_c) * np_cells - e * (np_cells / (np_cells + K_p)) * nTA
    return dnp_dt

In [3]:
def cart_activation(np_cells, nTA, nTN, params):
    """
    Calculates the derivative of activated CAR T-cells.
    """
    r_TA, K_r, k_A, K_A, l_TA = params
    dnTA_dt = r_TA * (np_cells / (np_cells + K_r)) * nTA + k_A * (np_cells / (np_cells + K_A)) * nTN - l_TA * nTA
    return dnTA_dt

In [4]:
def non_activated(np_cells, nTN, params):
    """
    Calculates the derivative of non-activated CAR T-cells.
    """
    k_A, K_A, l_TN = params
    dnTN_dt = -k_A * (np_cells / (np_cells + K_A)) * nTN - l_TN * nTN
    return dnTN_dt

In [5]:
def relapse(np_cells, nN, nTA, params):
    """
    Calculates the derivative of CD19⁻ tumor cells.
    """
    r_N, n_c, k_m, e, k_b, K_N = params
    dnN_dt = r_N * (1 - nN / n_c) * nN + k_m * np_cells - (e / k_b) * (nN / (nN + K_N)) * nTA
    return dnN_dt

In [6]:
# Combine the system of equations
def car_t_cell_model(y, t, params):
    np_cells, nTA, nTN, nN = y
    
    # Unpack parameters for each function
    params_cd19 = params[:4]              # [eta_p, n_c, e, K_p]
    params_activation = params[4:9]       # [r_TA, K_r, k_A, K_A, l_TA]
    params_non_activated = params[9:12]   # [k_A, K_A, l_TN]
    params_relapse = params[12:]          # [eta_N, n_c, k_m, e, k_b, K_N]
    
    # Compute each derivative
    dnp_dt = cd19_b_all_cell(np_cells, nTA, params_cd19)
    dnTA_dt = cart_activation(np_cells, nTA, nTN, params_activation)
    dnTN_dt = non_activated(np_cells, nTN, params_non_activated)
    dnN_dt = relapse(np_cells, nN, nTA, params_relapse)
    
    return [dnp_dt, dnTA_dt, dnTN_dt, dnN_dt]

In [7]:
def simulate_car_t_cell_model(y0, params, t):
    """ Simulates the CAR T-cell therapy model over a specified time period. """
    results = odeint(car_t_cell_model, y0, t, args=(params,))
    return results